In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from scipy.stats import uniform, randint

In [2]:
attributes = pd.read_csv('../data/fit_frenchie_attributes.csv')
attributes = attributes.astype(str)
attributes['nft_id'] = attributes['nft_id'].astype(int)

trades = pd.read_csv('../data/fit_frenchie_trades.csv')
trades = trades.astype(float)
trades['nft_id'] = trades['nft_id'].astype(int)

nft_1046 = pd.read_csv('../data/fit_frenchie_1046_attributes_freq.csv')

df = pd.read_csv('../data/fit_frenchie_trades_attributes_freq.csv')
df['normalized_trade_price'] = df['trade_price'] / df['floor_price']

nft_1046['floor_price'] = df['floor_price'].tail(1).values[0]

features = ['Accessories', 'Background', 'Body', 'Eyes', 'Hat', 'Mouth']

df

,nft_id,trade_price,floor_price,Accessories,Background,Body,Eyes,Hat,Mouth,Accessories_Background_freq,...,Eyes_Hat_freq,Eyes_Mouth_freq,Hat_Mouth_freq,Mouth_freq,Accessories_freq,Background_freq,Body_freq,Eyes_freq,Hat_freq,normalized_trade_price
0,1088,4.00,4.00,Tie,Midnight Sapphire Blue,Lilac,White Round Shades,No Hat,Sad,0.0073,...,0.0327,0.0120,0.0387,0.1180,0.0627,0.1167,0.0433,0.0987,0.3147,1.000000
1,1088,5.16,5.16,Tie,Midnight Sapphire Blue,Lilac,White Round Shades,No Hat,Sad,0.0073,...,0.0327,0.0120,0.0387,0.1180,0.0627,0.1167,0.0433,0.0987,0.3147,1.000000
2,1088,8.15,8.15,Tie,Midnight Sapphire Blue,Lilac,White Round Shades,No Hat,Sad,0.0073,...,0.0327,0.0120,0.0387,0.1180,0.0627,0.1167,0.0433,0.0987,0.3147,1.000000
3,1088,4.14,4.14,Tie,Midnight Sapphire Blue,Lilac,White Round Shades,No Hat,Sad,0.0073,...,0.0327,0.0120,0.0387,0.1180,0.0627,0.1167,0.0433,0.0987,0.3147,1.000000
4,1065,4.00,4.00,NaN,Spiced Orange,Tan,White Shades,Headphones,Toothy,0.0527,...,0.0087,0.0300,0.0087,0.1853,0.4247,0.1180,0.1553,0.1480,0.0500,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,36,1.51,0.95,Wings,Lavender Dusk Purple,Tan,Laser Eyes,No Hat,Sad,0.0013,...,0.0060,0.0013,0.0387,0.1180,0.0193,0.1187,0.1553,0.0167,0.3147,1.589474
1996,1028,0.94,0.94,Red Bandana,Forest Moss Green,Black - Overweight,Red Round Shades,Cylinder,Toothy,0.0167,...,0.0060,0.0127,0.0107,0.1853,0.0920,0.1247,0.1773,0.0587,0.0493,1.000000
1997,59,0.94,0.94,NaN,Sunlit Gold,Tan,Drowsy,Party Hat,Drooling,0.0673,...,0.0027,0.0053,0.0040,0.0700,0.4247,0.1500,0.1553,0.0407,0.0500,1.000000
1998,614,0.91,0.91,Neck Tattoo,Lavender Dusk Purple,Black,White Round Shades,Slickback,Toothy,0.0047,...,0.0033,0.0147,0.0107,0.1853,0.0300,0.1187,0.2333,0.0987,0.0493,1.000000


In [3]:
data = pd.get_dummies(df, columns=features)
data

,nft_id,trade_price,floor_price,Accessories_Background_freq,Accessories_Body_freq,Accessories_Eyes_freq,Accessories_Hat_freq,Accessories_Mouth_freq,Background_Body_freq,Background_Eyes_freq,...,Mouth_Bubble Gum,Mouth_Drooling,Mouth_Feral,Mouth_Happy,Mouth_Mustache,Mouth_Panting,Mouth_Sad,Mouth_Surprised,Mouth_Toothy,Mouth_Vampire
0,1088,4.00,4.00,0.0073,0.0053,0.0047,0.0240,0.0087,0.0053,0.0127,...,False,False,False,False,False,False,True,False,False,False
1,1088,5.16,5.16,0.0073,0.0053,0.0047,0.0240,0.0087,0.0053,0.0127,...,False,False,False,False,False,False,True,False,False,False
2,1088,8.15,8.15,0.0073,0.0053,0.0047,0.0240,0.0087,0.0053,0.0127,...,False,False,False,False,False,False,True,False,False,False
3,1088,4.14,4.14,0.0073,0.0053,0.0047,0.0240,0.0087,0.0053,0.0127,...,False,False,False,False,False,False,True,False,False,False
4,1065,4.00,4.00,0.0527,0.0660,0.0660,0.0220,0.0740,0.0207,0.0147,...,False,False,False,False,False,False,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,36,1.51,0.95,0.0013,0.0033,0.0013,0.0040,0.0027,0.0180,0.0033,...,False,False,False,False,False,False,True,False,False,False
1996,1028,0.94,0.94,0.0167,0.0180,0.0060,0.0067,0.0133,0.0260,0.0067,...,False,False,False,False,False,False,False,False,True,False
1997,59,0.94,0.94,0.0673,0.0660,0.0233,0.0247,0.0320,0.0267,0.0047,...,False,True,False,False,False,False,False,False,False,False
1998,614,0.91,0.91,0.0047,0.0087,0.0040,0.0027,0.0027,0.0300,0.0100,...,False,False,False,False,False,False,False,False,True,False


In [6]:
import autosklearn.regression

X = data.drop(['nft_id', 'normalized_trade_price'], axis=1)
y = data['normalized_trade_price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)


automodel = autosklearn.regression.AutoSklearnRegressor(
    time_left_for_this_task=20*60*60,  # Total time for optimization in seconds
    per_run_time_limit=60*60,        # Time limit for each model in seconds
    n_jobs=-1,                      # Use all available cores
    seed=42
)

automodel.fit(X_train, y_train)

y_pred = automodel.predict(X)

y_pred = pd.DataFrame(y_pred, columns=['Predicted'], index=X.index)
y = pd.DataFrame(y)
y.columns = ['Actual']

results = y.reset_index().merge(y_pred.reset_index(), on='index')

results['Difference']  = results['Actual'] - results['Predicted']
results['Difference%'] = results['Difference'] / results['Actual']
results['Difference**2'] = results['Difference'] ** 2

print(f"MSE  = {results['Difference**2'].mean():.2f}")
print(f"RMSE = {np.sqrt(results['Difference**2'].mean()):.2f}")
print(f"MAE  = {results['Difference'].abs().mean():.2f}")
print(f"MAPE = {results['Difference%'].abs().mean():.2%}")

results


Process pynisher function call:
Traceback (most recent call last):
  File "/spare/local/amorin/.conda/envs/ocean/lib/python3.8/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/spare/local/amorin/.conda/envs/ocean/lib/python3.8/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/spare/local/amorin/.conda/envs/ocean/lib/python3.8/site-packages/pynisher/limit_function_call.py", line 133, in subprocess_func
    return_value = ((func(*args, **kwargs), 0))
  File "/spare/local/amorin/.conda/envs/ocean/lib/python3.8/site-packages/autosklearn/smbo.py", line 160, in _calculate_metafeatures_encoded
    result = calculate_all_metafeatures_encoded_labels(
  File "/spare/local/amorin/.conda/envs/ocean/lib/python3.8/site-packages/autosklearn/metalearning/metafeatures/metafeatures.py", line 1115, in calculate_all_metafeatures_encoded_labels
    return calculate_all_metafeatures(
  File "/spare/local/amorin/.conda/envs/ocean/

[WARNING] [2024-09-13 17:53:52,436:Client-EnsembleBuilder] No runs were available to build an ensemble from
[WARNING] [2024-09-13 17:53:52,907:Client-EnsembleBuilder] No runs were available to build an ensemble from
[WARNING] [2024-09-13 17:53:53,217:Client-EnsembleBuilder] No runs were available to build an ensemble from
[WARNING] [2024-09-13 17:53:53,673:Client-EnsembleBuilder] No runs were available to build an ensemble from
[WARNING] [2024-09-13 17:53:54,285:Client-EnsembleBuilder] No runs were available to build an ensemble from
[WARNING] [2024-09-13 17:53:55,004:Client-EnsembleBuilder] No runs were available to build an ensemble from
[WARNING] [2024-09-13 17:53:55,232:Client-EnsembleBuilder] No runs were available to build an ensemble from
[WARNING] [2024-09-13 17:53:55,639:Client-EnsembleBuilder] No runs were available to build an ensemble from
[WARNING] [2024-09-13 17:53:58,908:Client-EnsembleBuilder] No runs were available to build an ensemble from
[WARNING] [2024-09-13 17:54:

In [ ]:
X.dtypes

NameError: name 'X' is not defined

In [ ]:
nft_1046_hot = pd.get_dummies(pd.concat([df.drop(['trade_price'], axis=1),nft_1046]), columns=features).tail(1)
nft_1046_hot = nft_1046_hot[X_train.columns]

y_1046 = automodel.predict(nft_1046_hot)
price_predicted = y_1046[0]
print(f"Predicted Normalized Price for the NFT 1046: {price_predicted:.2f}")
print(f"Latest floor price for the NFT 1046: {nft_1046['floor_price'].values[0]:.2f}")
print(f"Predicted Price for the NFT 1046: {price_predicted * nft_1046['floor_price'].values[0]:.2f}")

NameError: name 'pd' is not defined

In [ ]:
# Predicted Normalized Price for the NFT 1046: 7.11
# Latest floor price for the NFT 1046: 0.91
# Predicted Price for the NFT 1046: 6.47